### Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall

# 재현성 평가 (릴리번스, 앙상블)

In [ ]:
import os
import pandas as pd
import time
import warnings
from datasets import Dataset
from ragas import evaluate, RunConfig

# [설정] GPT 및 Ragas 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings('ignore')

# ==========================================
# 1. 환경 설정 (OpenAI API Key 및 경로)
# ==========================================
# ★★★ 여기에 실제 OpenAI API Key를 입력하세요 ★★★
os.environ["OPENAI_API_KEY"] = "" 

# GT 파일 경로 (기존 유지)
GT_PATH = r""

# [수정됨] 평가할 모델 및 경로 설정
MODELS_INFO = {
    "relevance-rag": {
        "ans_dir": r"",
        "ctx_dir": r""
    },
    "ensemble-rag": {
        "ans_dir": r"",
        "ctx_dir": r""
    }
}

TARGET_COLS = [
    "Particle size", "Particle shape", "Particle distribution", 
    "Coating layer characteristics", "Crystal structure and lattice characteristics"
]

# ==========================================
# 2. 모델 및 메트릭 설정
# ==========================================
# gpt-4.1-mini는 API명이 아니므로 gpt-4o-mini로 사용합니다.
gpt_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) 
gpt_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

ragas_llm = LangchainLLMWrapper(gpt_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(gpt_embeddings)

faithfulness_metric = Faithfulness()
answer_correctness_metric = AnswerCorrectness()
answer_relevancy_metric = AnswerRelevancy()
context_precision_metric = ContextPrecision()
context_recall_metric = ContextRecall()

my_metrics = [
    faithfulness_metric, 
    answer_correctness_metric, 
    answer_relevancy_metric, 
    context_precision_metric, 
    context_recall_metric
]

run_config = RunConfig(max_retries=10)

# ==========================================
# 3. 유틸리티 함수
# ==========================================
def load_context(context_dir, paper_id):
    file_path = os.path.join(context_dir, f"{paper_id}.txt")
    if not os.path.exists(file_path):
        return [""]
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().replace("context: |-", "").strip()
            return [content]
    except:
        return [""]

# ==========================================
# 4. 메인 실행 루프
# ==========================================
all_results = []
print("=== GPT 평가 시작 (Rule 1, 2, 3 적용) ===")

# GT 로드
try:
    df_gt = pd.read_csv(GT_PATH, encoding='cp949')
except UnicodeDecodeError:
    df_gt = pd.read_csv(GT_PATH, encoding='euc-kr')

df_gt['Paper ID'] = df_gt['Paper ID'].astype(str).str.replace(".0", "", regex=False)

# [수정됨] 2개 모델, 각각 1~10회차 순회
for model_name, paths in MODELS_INFO.items():
    for i in range(1, 11): # 1부터 10까지
        print(f"\nProcessing Model: {model_name} | Iteration: {i} ...")

        # Answer 경로 세팅 (ex: relevance-rag_1_resampling.xlsx)
        ans_filename = f"{model_name}_{i}_resampling.xlsx"
        ans_path = os.path.join(paths["ans_dir"], ans_filename)
        
        # Context 경로 세팅 (ex: ...\Relevance-rag\1 )
        context_dir = os.path.join(paths["ctx_dir"], str(i))

        if not os.path.exists(ans_path):
            print(f"Skipping {model_name}_{i}: Answer file not found at {ans_path}")
            continue

        if not os.path.exists(context_dir):
            print(f"Warning: Context folder not found at {context_dir}. Contexts will be empty.")

        # Answer 로드 (xlsx 파일이므로 read_excel 사용)
        try:
            df_ans = pd.read_excel(ans_path)
        except Exception as e:
            print(f"Error reading {ans_path}: {e}")
            continue

        df_ans['Paper ID'] = df_ans['Paper ID'].astype(str).str.replace(".0", "", regex=False)
        
        merged_df = pd.merge(df_gt, df_ans, on=['Paper ID', 'Sample'], how='inner', suffixes=('_gt', '_ans'))

        ragas_data = []
        
        for idx, row in merged_df.iterrows():
            paper_id = row['Paper ID']
            sample_name = row['Sample']
            contexts = load_context(context_dir, paper_id)

            for col in TARGET_COLS:
                question = f"What is the {col} of {sample_name}?"
                
                # -------------------------------------------------------
                # [Rule 2] 데이터 전처리 로직 (빈칸 채우기)
                # -------------------------------------------------------
                
                # 1. Ground Truth 처리
                raw_gt = row.get(f'{col}_gt')
                if pd.isna(raw_gt) or str(raw_gt).strip() == "":
                    val_gt = "Information not found in the context."
                else:
                    val_gt = str(raw_gt).strip()
                
                # 2. Answer 처리
                raw_ans = row.get(f'{col}_ans')
                is_ans_not_found = False 

                if pd.isna(raw_ans) or str(raw_ans).strip() == "":
                    val_ans = "Information not found in the context."
                elif "not found" in str(raw_ans).lower():
                    val_ans = "not found"
                    is_ans_not_found = True
                else:
                    val_ans = str(raw_ans).strip()

                # -------------------------------------------------------
                # [Rule 3 준비] 둘 다 "Information not found..." 인지 확인
                # -------------------------------------------------------
                target_phrase = "Information not found in the context."
                is_both_not_found = (val_gt.strip() == target_phrase) and (val_ans.strip() == target_phrase)

                ragas_data.append({
                    "question": question,
                    "answer": val_ans,
                    "contexts": contexts,
                    "ground_truth": val_gt, 
                    "metadata": {
                        "Paper ID": paper_id, 
                        "Type": col,
                        "Model": model_name,
                        "Iteration": i,
                        "Is_Ans_NotFound": is_ans_not_found,  
                        "Is_Both_NotFound": is_both_not_found 
                    }
                })

        if not ragas_data:
            print(f"No valid data to evaluate for {model_name}_{i}. Skipping...")
            continue

        dataset = Dataset.from_list(ragas_data)
        
        try:
            result = evaluate(
                dataset=dataset,
                metrics=my_metrics,
                llm=ragas_llm,
                embeddings=ragas_embeddings,
                run_config=run_config
            )
            res_df = result.to_pandas()
            
        except Exception as e:
            print(f"Error during evaluation of {model_name}_{i}: {e}")
            import traceback
            traceback.print_exc()
            continue

        # ==========================================
        # ⚡ 점수 처리 로직
        # ==========================================
        # 메타데이터 복구
        meta_list = [item['metadata'] for item in ragas_data]
        res_df['Is_Ans_NotFound'] = [m['Is_Ans_NotFound'] for m in meta_list]
        res_df['Is_Both_NotFound'] = [m['Is_Both_NotFound'] for m in meta_list]
        res_df['Paper ID'] = [m['Paper ID'] for m in meta_list]
        res_df['Type'] = [m['Type'] for m in meta_list]
        res_df['Model'] = [m['Model'] for m in meta_list]
        res_df['Iteration'] = [m['Iteration'] for m in meta_list]

        def apply_penalty_to_row(row):
            # =======================================================
            # Rule 3: [최우선] 둘 다 "정보 없음"이면 정답이므로 만점(1.0) 처리
            # =======================================================
            if row['Is_Both_NotFound']:
                row['faithfulness'] = 1.0
                row['answer_correctness'] = 1.0
                row['answer_relevancy'] = 1.0
                row['context_precision'] = 1.0
                row['context_recall'] = 1.0
                return row  

            # =======================================================
            # Rule 1: [차순위] 모델만 못 찾았으면 0점 처리
            # =======================================================
            if row['Is_Ans_NotFound']:
                row['faithfulness'] = 0.0
                row['answer_correctness'] = 0.0
                row['answer_relevancy'] = 0.0
                row['context_precision'] = 0.0
                row['context_recall'] = 0.0
                
            return row

        # 1. 0점 / 만점 규칙 적용
        res_df = res_df.apply(apply_penalty_to_row, axis=1)

        # 2. 불필요한 열 삭제
        drop_cols = ['Is_Ans_NotFound', 'Is_Both_NotFound']
        res_df = res_df.drop(columns=[c for c in drop_cols if c in res_df.columns])
        
        # 3. 열 순서 재배치 (Model, Iteration 추가)
        target_order = [
            'Model', 'Iteration', 'Paper ID', 'Type', 
            'user_input', 
            'retrieved_contexts', 
            'response', 
            'reference', 
            'faithfulness',
            'answer_relevancy', 
            'answer_correctness',
            'context_precision',
            'context_recall'
        ]
        
        final_cols = [col for col in target_order if col in res_df.columns]
        remaining_cols = [col for col in res_df.columns if col not in final_cols]
        
        res_df = res_df[final_cols + remaining_cols]

        # 결과 저장 (개별 저장)
        all_results.append(res_df)
        save_name = f"Result_{model_name}_{i}.xlsx"
        res_df.to_excel(save_name, index=False)
        print(f"Done: {model_name}_{i} -> Saved to {save_name}")
        time.sleep(1)

# [선택사항] 20개의 결과를 하나의 엑셀 파일로 묶어서 저장하고 싶을 경우 주석 해제
# if all_results:
#    final_df = pd.concat(all_results, ignore_index=True)
#    save_path = os.path.join(r"C:\Users\LECS\Desktop\JSY\10. AIFFEL\VOLTAI-main\code\JSY\Reproducibility_test\output", "Final_All_Results.xlsx")
#    final_df.to_excel(save_path, index=False)
#    print(f"\n모든 평가 완료! 통합 결과 저장됨:\n{save_path}")

print("\n=== 모든 평가 루프 종료 ===")

C:\Users\LECS\AppData\Local\Temp\ipykernel_30760\473872910.py:10: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\LECS\AppData\Local\Temp\ipykernel_30760\473872910.py:10: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\LECS\AppData\Local\Temp\ipykernel_30760\473872910.py:10: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collec

=== GPT 평가 시작 (Rule 1, 2, 3 적용) ===

Processing Model: relevance-rag | Iteration: 1 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: relevance-rag_1 -> Saved to Result_relevance-rag_1.xlsx

Processing Model: relevance-rag | Iteration: 2 ...


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: ensemble-rag_9 -> Saved to Result_ensemble-rag_9.xlsx

Processing Model: ensemble-rag | Iteration: 10 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: ensemble-rag_10 -> Saved to Result_ensemble-rag_10.xlsx

=== 모든 평가 루프 종료 ===


# 재현성 평가(멀티에이전트)

In [ ]:
import os
import pandas as pd
import time
import warnings
from datasets import Dataset
from ragas import evaluate, RunConfig

# [설정] GPT 및 Ragas 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings('ignore')

# ==========================================
# 1. 환경 설정 (OpenAI API Key 및 경로)
# ==========================================
# ★★★ 여기에 실제 OpenAI API Key를 입력하세요 ★★★
os.environ["OPENAI_API_KEY"] = "" 

# GT 파일 경로 (기존 유지)
GT_PATH = r""

# [수정됨] Multiagent-rag 1개 모델만 평가하도록 설정
MODELS_INFO = {
    "multiagent-rag": {
        "ans_dir": r"",
        "ctx_dir": r""
    }
}

TARGET_COLS = [
    "Particle size", "Particle shape", "Particle distribution", 
    "Coating layer characteristics", "Crystal structure and lattice characteristics"
]

# ==========================================
# 2. 모델 및 메트릭 설정
# ==========================================
gpt_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) 
gpt_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

ragas_llm = LangchainLLMWrapper(gpt_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(gpt_embeddings)

faithfulness_metric = Faithfulness()
answer_correctness_metric = AnswerCorrectness()
answer_relevancy_metric = AnswerRelevancy()
context_precision_metric = ContextPrecision()
context_recall_metric = ContextRecall()

my_metrics = [
    faithfulness_metric, 
    answer_correctness_metric, 
    answer_relevancy_metric, 
    context_precision_metric, 
    context_recall_metric
]

run_config = RunConfig(max_retries=10)

# ==========================================
# 3. 유틸리티 함수
# ==========================================
def load_context(context_dir, paper_id):
    file_path = os.path.join(context_dir, f"{paper_id}.txt")
    if not os.path.exists(file_path):
        return [""]
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().replace("context: |-", "").strip()
            return [content]
    except:
        return [""]

# ==========================================
# 4. 메인 실행 루프
# ==========================================
all_results = []
print("=== GPT 평가 시작 (Rule 1, 2, 3 적용) ===")

# GT 로드
try:
    df_gt = pd.read_csv(GT_PATH, encoding='cp949')
except UnicodeDecodeError:
    df_gt = pd.read_csv(GT_PATH, encoding='euc-kr')

df_gt['Paper ID'] = df_gt['Paper ID'].astype(str).str.replace(".0", "", regex=False)

# Multiagent 모델 1~10회차 순회
for model_name, paths in MODELS_INFO.items():
    for i in range(1, 11): # 1부터 10까지
        print(f"\nProcessing Model: {model_name} | Iteration: {i} ...")

        # Answer 경로 세팅 
        ans_filename = f"{model_name}_{i}_resampling.xlsx"
        ans_path = os.path.join(paths["ans_dir"], ans_filename)
        
        # Context 경로 세팅 
        context_dir = os.path.join(paths["ctx_dir"], str(i))

        if not os.path.exists(ans_path):
            print(f"Skipping {model_name}_{i}: Answer file not found at {ans_path}")
            continue

        if not os.path.exists(context_dir):
            print(f"Warning: Context folder not found at {context_dir}. Contexts will be empty.")

        # Answer 로드 
        try:
            df_ans = pd.read_excel(ans_path)
        except Exception as e:
            print(f"Error reading {ans_path}: {e}")
            continue

        df_ans['Paper ID'] = df_ans['Paper ID'].astype(str).str.replace(".0", "", regex=False)
        
        merged_df = pd.merge(df_gt, df_ans, on=['Paper ID', 'Sample'], how='inner', suffixes=('_gt', '_ans'))

        ragas_data = []
        
        for idx, row in merged_df.iterrows():
            paper_id = row['Paper ID']
            sample_name = row['Sample']
            contexts = load_context(context_dir, paper_id)

            for col in TARGET_COLS:
                question = f"What is the {col} of {sample_name}?"
                
                # -------------------------------------------------------
                # [Rule 2] 데이터 전처리 로직 (빈칸 채우기)
                # -------------------------------------------------------
                
                # 1. Ground Truth 처리
                raw_gt = row.get(f'{col}_gt')
                if pd.isna(raw_gt) or str(raw_gt).strip() == "":
                    val_gt = "Information not found in the context."
                else:
                    val_gt = str(raw_gt).strip()
                
                # 2. Answer 처리
                raw_ans = row.get(f'{col}_ans')
                is_ans_not_found = False 

                if pd.isna(raw_ans) or str(raw_ans).strip() == "":
                    val_ans = "Information not found in the context."
                elif "not found" in str(raw_ans).lower():
                    val_ans = "not found"
                    is_ans_not_found = True
                else:
                    val_ans = str(raw_ans).strip()

                # -------------------------------------------------------
                # [Rule 3 준비] 둘 다 "Information not found..." 인지 확인
                # -------------------------------------------------------
                target_phrase = "Information not found in the context."
                is_both_not_found = (val_gt.strip() == target_phrase) and (val_ans.strip() == target_phrase)

                ragas_data.append({
                    "question": question,
                    "answer": val_ans,
                    "contexts": contexts,
                    "ground_truth": val_gt, 
                    "metadata": {
                        "Paper ID": paper_id, 
                        "Type": col,
                        "Model": model_name,
                        "Iteration": i,
                        "Is_Ans_NotFound": is_ans_not_found,  
                        "Is_Both_NotFound": is_both_not_found 
                    }
                })

        if not ragas_data:
            print(f"No valid data to evaluate for {model_name}_{i}. Skipping...")
            continue

        dataset = Dataset.from_list(ragas_data)
        
        try:
            result = evaluate(
                dataset=dataset,
                metrics=my_metrics,
                llm=ragas_llm,
                embeddings=ragas_embeddings,
                run_config=run_config
            )
            res_df = result.to_pandas()
            
        except Exception as e:
            print(f"Error during evaluation of {model_name}_{i}: {e}")
            import traceback
            traceback.print_exc()
            continue

        # ==========================================
        # ⚡ 점수 처리 로직
        # ==========================================
        # 메타데이터 복구
        meta_list = [item['metadata'] for item in ragas_data]
        res_df['Is_Ans_NotFound'] = [m['Is_Ans_NotFound'] for m in meta_list]
        res_df['Is_Both_NotFound'] = [m['Is_Both_NotFound'] for m in meta_list]
        res_df['Paper ID'] = [m['Paper ID'] for m in meta_list]
        res_df['Type'] = [m['Type'] for m in meta_list]
        res_df['Model'] = [m['Model'] for m in meta_list]
        res_df['Iteration'] = [m['Iteration'] for m in meta_list]

        def apply_penalty_to_row(row):
            # =======================================================
            # Rule 3: [최우선] 둘 다 "정보 없음"이면 정답이므로 만점(1.0) 처리
            # =======================================================
            if row['Is_Both_NotFound']:
                row['faithfulness'] = 1.0
                row['answer_correctness'] = 1.0
                row['answer_relevancy'] = 1.0
                row['context_precision'] = 1.0
                row['context_recall'] = 1.0
                return row  

            # =======================================================
            # Rule 1: [차순위] 모델만 못 찾았으면 0점 처리
            # =======================================================
            if row['Is_Ans_NotFound']:
                row['faithfulness'] = 0.0
                row['answer_correctness'] = 0.0
                row['answer_relevancy'] = 0.0
                row['context_precision'] = 0.0
                row['context_recall'] = 0.0
                
            return row

        # 1. 0점 / 만점 규칙 적용
        res_df = res_df.apply(apply_penalty_to_row, axis=1)

        # 2. 불필요한 열 삭제
        drop_cols = ['Is_Ans_NotFound', 'Is_Both_NotFound']
        res_df = res_df.drop(columns=[c for c in drop_cols if c in res_df.columns])
        
        # 3. 열 순서 재배치 (Model, Iteration 추가)
        target_order = [
            'Model', 'Iteration', 'Paper ID', 'Type', 
            'user_input', 
            'retrieved_contexts', 
            'response', 
            'reference', 
            'faithfulness',
            'answer_relevancy', 
            'answer_correctness',
            'context_precision',
            'context_recall'
        ]
        
        final_cols = [col for col in target_order if col in res_df.columns]
        remaining_cols = [col for col in res_df.columns if col not in final_cols]
        
        res_df = res_df[final_cols + remaining_cols]

        # 결과 저장 (개별 저장)
        all_results.append(res_df)
        save_name = f"Result_{model_name}_{i}.xlsx"
        res_df.to_excel(save_name, index=False)
        print(f"Done: {model_name}_{i} -> Saved to {save_name}")
        time.sleep(1)

print("\n=== 모든 평가 루프 종료 ===")

=== GPT 평가 시작 (Rule 1, 2, 3 적용) ===

Processing Model: multiagent-rag | Iteration: 1 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: multiagent-rag_1 -> Saved to Result_multiagent-rag_1.xlsx

Processing Model: multiagent-rag | Iteration: 2 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: multiagent-rag_9 -> Saved to Result_multiagent-rag_9.xlsx

Processing Model: multiagent-rag | Iteration: 10 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: multiagent-rag_10 -> Saved to Result_multiagent-rag_10.xlsx

=== 모든 평가 루프 종료 ===


# 하이퍼파라미터 평가

In [ ]:
import os
import pandas as pd
import time
import warnings
from datasets import Dataset
from ragas import evaluate, RunConfig

# [설정] GPT 및 Ragas 임포트
# AnswerRelevancy 제거됨
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings('ignore')

# ==========================================
# 1. 환경 설정 (OpenAI API Key)
# ==========================================
# ★★★ 여기에 실제 OpenAI API Key를 입력하세요 ★★★
os.environ["OPENAI_API_KEY"] = "" 

# 파일 경로 설정
BASE_PATH = r""
GT_PATH = os.path.join(BASE_PATH, "260130_GT.csv")

# 하이퍼파라미터 조합 (테스트용 1개, 필요시 전체 주석 해제)
CHUNK_SIZES = [512, 1024, 2048] # [512, 1024, 2048]
CHUNK_OVERLAPS = [30, 50, 100] # [30, 50, 100]
TOP_KS = [3, 6, 9] # [3, 6, 9]

TARGET_COLS = [
    "Particle size", "Particle shape", "Particle distribution", 
    "Coating layer characteristics", "Crystal structure and lattice characteristics"
]

# ==========================================
# 2. 모델 및 메트릭 설정
# ==========================================
gpt_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0) # gpt-4.1-mini는 존재하지 않아 gpt-4o-mini로 기재했습니다 (확인 필요)
gpt_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

ragas_llm = LangchainLLMWrapper(gpt_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(gpt_embeddings)

faithfulness_metric = Faithfulness()
answer_correctness_metric = AnswerCorrectness()
answer_relevancy_metric = AnswerRelevancy()
context_precision_metric = ContextPrecision()
context_recall_metric = ContextRecall()

# AnswerRelevancy 제외하고 리스트 구성
my_metrics = [faithfulness_metric, answer_correctness_metric, answer_relevancy_metric, context_precision_metric, context_recall_metric]

# API 속도 조절
run_config = RunConfig(max_retries=10)

# ==========================================
# 3. 유틸리티 함수
# ==========================================
def load_context(context_dir, paper_id):
    file_path = os.path.join(context_dir, f"{paper_id}.txt")
    if not os.path.exists(file_path):
        return [""]
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().replace("context: |-", "").strip()
            return [content]
    except:
        return [""]

# ==========================================
# 4. 메인 실행 루프
# ==========================================
all_results = []
print("=== GPT 평가 시작 (AnswerRelevancy 제외) ===")

# GT 로드
try:
    df_gt = pd.read_csv(GT_PATH, encoding='cp949')
except UnicodeDecodeError:
    df_gt = pd.read_csv(GT_PATH, encoding='euc-kr')

df_gt['Paper ID'] = df_gt['Paper ID'].astype(str).str.replace(".0", "", regex=False)

for chunk in CHUNK_SIZES:
    for overlap in CHUNK_OVERLAPS:
        for k in TOP_KS:
            combination = f"{chunk}_{overlap}_{k}"
            print(f"\nProcessing: {combination} ...")

            ans_path = os.path.join(BASE_PATH, "리샘플링_CSV", f"relevance-rag({combination})_resampling.csv")
            context_dir = os.path.join(BASE_PATH, "카테고리3번_CONTEXT", combination)

            if not os.path.exists(ans_path):
                print(f"Skipping {combination}: Answer file not found.")
                continue

            # Answer 로드
            try:
                df_ans = pd.read_csv(ans_path, encoding='utf-8')
            except UnicodeDecodeError:
                df_ans = pd.read_csv(ans_path, encoding='cp949')

            df_ans['Paper ID'] = df_ans['Paper ID'].astype(str).str.replace(".0", "", regex=False)
            
            merged_df = pd.merge(df_gt, df_ans, on=['Paper ID', 'Sample'], how='inner', suffixes=('_gt', '_ans'))

            ragas_data = []
            
            for idx, row in merged_df.iterrows():
                paper_id = row['Paper ID']
                sample_name = row['Sample']
                contexts = load_context(context_dir, paper_id)

                for col in TARGET_COLS:
                    question = f"What is the {col} of {sample_name}?"
                    
                    # -------------------------------------------------------
                    # 데이터 전처리 로직
                    # -------------------------------------------------------
                    
                    # 1. Ground Truth 처리
                    raw_gt = row[f'{col}_gt']
                    if pd.isna(raw_gt) or str(raw_gt).strip() == "":
                        val_gt = "Information not found in the context."
                    else:
                        val_gt = str(raw_gt).strip()
                    
                    # 2. Answer 처리
                    raw_ans = row[f'{col}_ans']
                    is_ans_not_found = False 

                    if pd.isna(raw_ans) or str(raw_ans).strip() == "":
                        val_ans = "Information not found in the context."
                    elif "not found" in str(raw_ans).lower():
                        val_ans = "not found"
                        is_ans_not_found = True
                    else:
                        val_ans = str(raw_ans).strip()

                    # [위치] ragas_data.append 직전 (val_gt, val_ans 처리 로직 바로 아래)

                    # -------------------------------------------------------
                    # [추가] Rule 3: 둘 다 "Information not found..." 인지 확인
                    # -------------------------------------------------------
                    target_phrase = "Information not found in the context."

                    # 문자열이 정확히 일치하는지 확인 (공백 제거 후 비교)
                    is_both_not_found = (val_gt.strip() == target_phrase) and (val_ans.strip() == target_phrase)

                    ragas_data.append({
                        "question": question,
                        "answer": val_ans,
                        "contexts": contexts,
                        "ground_truth": val_gt, 
                        "metadata": {
                            "Paper ID": paper_id, 
                            "Type": col,
                            "Combination": combination,
                            "Is_Ans_NotFound": is_ans_not_found,  # 기존 Rule 1용 플래그
                            "Is_Both_NotFound": is_both_not_found # [New] Rule 3용 플래그
                        }
                    })

            if not ragas_data:
                continue

            dataset = Dataset.from_list(ragas_data)
            
            try:
                result = evaluate(
                    dataset=dataset,
                    metrics=my_metrics,
                    llm=ragas_llm,
                    embeddings=ragas_embeddings,
                    run_config=run_config
                )
                res_df = result.to_pandas()
                
            except Exception as e:
                print(f"Error during evaluation of {combination}: {e}")
                import traceback
                traceback.print_exc()
                continue

            # ==========================================
            # ⚡ [수정됨] 0점 처리 로직 (Relevancy 제외)
            # ==========================================
            # 메타데이터 복구
            meta_list = [item['metadata'] for item in ragas_data]
            res_df['Is_Ans_NotFound'] = [m['Is_Ans_NotFound'] for m in meta_list]
            res_df['Is_Both_NotFound'] = [m['Is_Both_NotFound'] for m in meta_list]
            res_df['Paper ID'] = [m['Paper ID'] for m in meta_list]
            res_df['Type'] = [m['Type'] for m in meta_list]
            res_df['Combination'] = combination



            def apply_penalty_to_row(row):
                # =======================================================
                # Rule 3: [최우선] 둘 다 "정보 없음"이면 정답이므로 만점(1.0) 처리
                # =======================================================
                if row['Is_Both_NotFound']:
                    row['faithfulness'] = 1.0
                    row['answer_correctness'] = 1.0
                    row['answer_relevancy'] = 1.0
                    row['context_precision'] = 1.0
                    row['context_recall'] = 1.0
                    return row  # 만점 처리 후 바로 종료 (아래 0점 로직 실행 안 함)

                # =======================================================
                # Rule 1: [차순위] 모델만 못 찾았으면(GT는 있는데) 0점 처리
                # =======================================================
                if row['Is_Ans_NotFound']:
                    row['faithfulness'] = 0.0
                    row['answer_correctness'] = 0.0
                    row['answer_relevancy'] = 0.0
                    row['context_precision'] = 0.0
                    row['context_recall'] = 0.0
                    
                return row

            # 1. 0점 페널티 적용
            res_df = res_df.apply(apply_penalty_to_row, axis=1)

            # 2. 불필요한 열 삭제
            if 'Is_Ans_NotFound' in res_df.columns:
                res_df = res_df.drop(columns=['Is_Ans_NotFound'])
            
            # 3. 열 순서 재배치 (AnswerRelevancy 제거됨)
            target_order = [
                'Combination', 'Paper ID', 'Type', 
                'user_input',           # question -> user_input
                'retrieved_contexts',   # contexts -> retrieved_contexts
                'response',             # answer -> response
                'reference',            # ground_truth -> reference
                'faithfulness',
                'answer_relevancy', 
                'answer_correctness',
                'context_precision',
                'context_recall'
            ]
            
            final_cols = [col for col in target_order if col in res_df.columns]
            remaining_cols = [col for col in res_df.columns if col not in final_cols]
            
            res_df = res_df[final_cols + remaining_cols]

            # 결과 저장
            all_results.append(res_df)
            res_df.to_excel(f"temp_result_{combination}.xlsx", index=False)
            print(f"Done: {combination}")
            time.sleep(1)

# 최종 저장
#if all_results:
#    final_df = pd.concat(all_results, ignore_index=True)
#    save_path = os.path.join(BASE_PATH, "Final_Ragas_Results_Penalty_Applied.xlsx")
#    final_df.to_excel(save_path, index=False)
#    print(f"\n모든 평가 완료! 결과 저장됨:\n{save_path}")

=== GPT 평가 시작 (AnswerRelevancy 제외) ===

Processing: 512_30_3 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: 512_30_3

Processing: 512_30_6 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: 2048_100_6

Processing: 2048_100_9 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: 2048_100_9


# LLM 모델 별 평가

In [ ]:
import os
import pandas as pd
import time
import warnings
from datasets import Dataset
from ragas import evaluate, RunConfig

# [설정] GPT 및 Ragas 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

warnings.filterwarnings('ignore')

# ==========================================
# 1. 환경 설정 (OpenAI API Key 및 경로)
# ==========================================
# ★★★ 여기에 실제 OpenAI API Key를 입력하세요 ★★★
os.environ["OPENAI_API_KEY"] = "" 

# -----------------------------------------------------------------------------------------
# [수정됨] 경로 설정
# -----------------------------------------------------------------------------------------
# GT 파일 경로 (기존 유지)
GT_PATH = r""

# Answer 파일들이 있는 폴더 경로
ANSWER_DIR = r""

# Context 폴더들이 있는 루트 경로
CONTEXT_DIR_ROOT = r""

# [수정됨] 평가할 모델 리스트 (파일명 및 폴더명으로 사용)
TARGET_MODELS = [
    "Gemini-3-flash-preview", 
    "Gemini-3-pro-preview", 
    "Gpt-4.1-mini", 
    "Gpt-5.2"
]

TARGET_COLS = [
    "Particle size", "Particle shape", "Particle distribution", 
    "Coating layer characteristics", "Crystal structure and lattice characteristics"
]

# ==========================================
# 2. 모델 및 메트릭 설정 (Evaluator)
# ==========================================
# 평가자(Judge) 모델 설정 (실제 API 호출용)
gpt_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # gpt-4.1-mini는 API명이 아니므로 gpt-4o-mini 사용
gpt_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

ragas_llm = LangchainLLMWrapper(gpt_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(gpt_embeddings)

# 메트릭 인스턴스 생성
faithfulness_metric = Faithfulness()
answer_correctness_metric = AnswerCorrectness()
answer_relevancy_metric = AnswerRelevancy()
context_precision_metric = ContextPrecision()
context_recall_metric = ContextRecall()

# 전체 메트릭 리스트
my_metrics = [
    faithfulness_metric, 
    answer_correctness_metric, 
    answer_relevancy_metric, 
    context_precision_metric, 
    context_recall_metric
]

# API 속도 조절
run_config = RunConfig(max_retries=10)

# ==========================================
# 3. 유틸리티 함수
# ==========================================
def load_context(context_dir, paper_id):
    file_path = os.path.join(context_dir, f"{paper_id}.txt")
    if not os.path.exists(file_path):
        return [""]
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().replace("context: |-", "").strip()
            return [content]
    except:
        return [""]

# ==========================================
# 4. 메인 실행 루프
# ==========================================
all_results = []
print("=== 모델별 평가 시작 (Rule 1, 2, 3 적용) ===")

# GT 로드
try:
    df_gt = pd.read_csv(GT_PATH, encoding='cp949')
except UnicodeDecodeError:
    df_gt = pd.read_csv(GT_PATH, encoding='euc-kr')
except FileNotFoundError:
    print(f"Error: GT 파일을 찾을 수 없습니다.\n경로 확인: {GT_PATH}")
    exit()

df_gt['Paper ID'] = df_gt['Paper ID'].astype(str).str.replace(".0", "", regex=False)

# [수정됨] 모델 리스트 반복
for model_name in TARGET_MODELS:
    print(f"\nProcessing Model: {model_name} ...")

    # 경로 구성
    ans_filename = f"{model_name}_리샘플링.xlsx"
    ans_path = os.path.join(ANSWER_DIR, ans_filename)
    context_dir = os.path.join(CONTEXT_DIR_ROOT, model_name) # 폴더명 = 모델명

    # 파일 존재 확인
    if not os.path.exists(ans_path):
        print(f"Skipping {model_name}: Answer file not found at {ans_path}")
        continue
    
    if not os.path.exists(context_dir):
        print(f"Warning: Context folder not found at {context_dir}. Contexts will be empty.")

    # Answer 로드
    try:
        df_ans = pd.read_excel(ans_path) # xlsx 파일이므로 read_excel 사용
    except Exception as e:
        print(f"Error reading {ans_path}: {e}")
        continue

    df_ans['Paper ID'] = df_ans['Paper ID'].astype(str).str.replace(".0", "", regex=False)
    
    # 데이터 병합
    merged_df = pd.merge(df_gt, df_ans, on=['Paper ID', 'Sample'], how='inner', suffixes=('_gt', '_ans'))

    ragas_data = []
    
    for idx, row in merged_df.iterrows():
        paper_id = row['Paper ID']
        sample_name = row['Sample']
        contexts = load_context(context_dir, paper_id)

        for col in TARGET_COLS:
            question = f"What is the {col} of {sample_name}?"
            
            # -------------------------------------------------------
            # [Rule 2] 빈칸 채우기 (데이터 전처리)
            # -------------------------------------------------------
            
            # 1. Ground Truth 처리
            raw_gt = row.get(f'{col}_gt') # .get()으로 안전하게 접근
            if pd.isna(raw_gt) or str(raw_gt).strip() == "":
                val_gt = "Information not found in the context."
            else:
                val_gt = str(raw_gt).strip()
            
            # 2. Answer 처리
            raw_ans = row.get(f'{col}_ans')
            is_ans_not_found = False 

            if pd.isna(raw_ans) or str(raw_ans).strip() == "":
                val_ans = "Information not found in the context."
            elif "not found" in str(raw_ans).lower():
                val_ans = "not found"
                is_ans_not_found = True
            else:
                val_ans = str(raw_ans).strip()

            # -------------------------------------------------------
            # [Rule 3 준비] 둘 다 "Information not found..." 인지 확인
            # -------------------------------------------------------
            target_phrase = "Information not found in the context."
            # 공백 제거 후 정확히 일치하는지 확인
            is_both_not_found = (val_gt.strip() == target_phrase) and (val_ans.strip() == target_phrase)

            ragas_data.append({
                "question": question,
                "answer": val_ans,
                "contexts": contexts,
                "ground_truth": val_gt, 
                "metadata": {
                    "Paper ID": paper_id, 
                    "Type": col,
                    "Model": model_name,
                    "Is_Ans_NotFound": is_ans_not_found,  # Rule 1용 플래그
                    "Is_Both_NotFound": is_both_not_found # Rule 3용 플래그
                }
            })

    if not ragas_data:
        print(f"No data to evaluate for {model_name}")
        continue

    # 데이터셋 생성 및 평가 실행
    dataset = Dataset.from_list(ragas_data)
    
    try:
        result = evaluate(
            dataset=dataset,
            metrics=my_metrics,
            llm=ragas_llm,
            embeddings=ragas_embeddings,
            run_config=run_config
        )
        res_df = result.to_pandas()
        
    except Exception as e:
        print(f"Error during evaluation of {model_name}: {e}")
        import traceback
        traceback.print_exc()
        continue

    # ==========================================
    # 점수 후처리 로직 (Rule 3 -> Rule 1)
    # ==========================================
    # 메타데이터 복구
    meta_list = [item['metadata'] for item in ragas_data]
    res_df['Is_Ans_NotFound'] = [m['Is_Ans_NotFound'] for m in meta_list]
    res_df['Is_Both_NotFound'] = [m['Is_Both_NotFound'] for m in meta_list]
    res_df['Paper ID'] = [m['Paper ID'] for m in meta_list]
    res_df['Type'] = [m['Type'] for m in meta_list]
    res_df['Model'] = model_name

    def apply_penalty_to_row(row):
        # =======================================================
        # [Rule 3: 최우선] 둘 다 "정보 없음"이면 정답이므로 만점(1.0) 처리
        # =======================================================
        if row['Is_Both_NotFound']:
            row['faithfulness'] = 1.0
            row['answer_correctness'] = 1.0
            row['answer_relevancy'] = 1.0
            row['context_precision'] = 1.0
            row['context_recall'] = 1.0
            return row  # 만점 처리 후 바로 종료

        # =======================================================
        # [Rule 1: 차순위] 모델만 못 찾았으면(GT는 있는데) 0점 처리
        # =======================================================
        if row['Is_Ans_NotFound']:
            row['faithfulness'] = 0.0
            row['answer_correctness'] = 0.0
            row['answer_relevancy'] = 0.0
            row['context_precision'] = 0.0
            row['context_recall'] = 0.0
            
        return row

    # 1. 점수 규칙 적용
    res_df = res_df.apply(apply_penalty_to_row, axis=1)

    # 2. 불필요한 열 삭제
    drop_cols = ['Is_Ans_NotFound', 'Is_Both_NotFound']
    res_df = res_df.drop(columns=[c for c in drop_cols if c in res_df.columns])
    
    # 3. 열 순서 재배치
    target_order = [
        'Model', 'Paper ID', 'Type', 
        'user_input', 
        'retrieved_contexts', 
        'response', 
        'reference', 
        'faithfulness',
        'answer_relevancy', 
        'answer_correctness',
        'context_precision',
        'context_recall'
    ]
    
    final_cols = [col for col in target_order if col in res_df.columns]
    remaining_cols = [col for col in res_df.columns if col not in final_cols]
    
    res_df = res_df[final_cols + remaining_cols]

    # 결과 저장
    all_results.append(res_df)
    save_name = f"Result_{model_name}.xlsx"
    res_df.to_excel(save_name, index=False)
    print(f"Done: {model_name} -> Saved to {save_name}")
    time.sleep(1)

print("\n=== 모든 모델 평가 완료 ===")

C:\Users\LECS\AppData\Local\Temp\ipykernel_28284\1265355964.py:10: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\LECS\AppData\Local\Temp\ipykernel_28284\1265355964.py:10: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import Faithfulness, AnswerCorrectness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\LECS\AppData\Local\Temp\ipykernel_28284\1265355964.py:10: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.col

=== 모델별 평가 시작 (Rule 1, 2, 3 적용) ===

Processing Model: Gemini-3-flash-preview ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: Gemini-3-flash-preview -> Saved to Result_Gemini-3-flash-preview.xlsx

Processing Model: Gemini-3-pro-preview ...


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: Gpt-4.1-mini -> Saved to Result_Gpt-4.1-mini.xlsx

Processing Model: Gpt-5.2 ...


Evaluating:   0%|          | 0/675 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done: Gpt-5.2 -> Saved to Result_Gpt-5.2.xlsx

=== 모든 모델 평가 완료 ===
